# 4 - CIDER T-cell Score vs RNA and ImmuneLENS Validation

**Purpose**: Validates CIDER T-cell Score against two orthogonal measures of
T-cell infiltration: RNA T-cell gene signatures and ImmuneLENS WGS-based
T-cell fraction estimates.

| | |
|---|---|
| **Inputs** | `CIDER_PROJ_DIR/ready.rds` |
| **Outputs** | `figs/tcell_validation_main.png`, `figs/tcell_validation_supplement.png` |

In [1]:
source(paste0(dirname(getwd()), "/helpers/map.r"))
source(paste0(dirname(getwd()), "/helpers/shortcuts.r"))
source(paste0(dirname(getwd()), "/helpers/config.r"))
source(paste0(dirname(getwd()), "/helpers/themes.r"))

# Ensure figures output directory exists
dir.create(paste0(CIDER_PROJ_DIR, "figs/"), showWarnings = FALSE, recursive = TRUE)

In [2]:
library(ggpubr)
library(patchwork)

# 0 - Read data

In [ ]:
# RNA validation uses all Hartwig patients with RNA-seq (broader validation).
# ImmuneLENS validation uses CPI patients only — ImmuneLENS was run on the
# ICI-treated cohort and non-CPI samples should not be included.
ready <-
  readRDS(paste0(CIDER_PROJ_DIR, "ready.rds")) |>
  mutate(rna = tcell_tpm)

# Long-format for RNA scatter plots
ready_rna <- ready |>
  transmute(sampleId, gp, rna, cider_score_depth_adj) |>
  drop_na(rna)

# ImmuneLENS: CPI patients only
ready_il <- readRDS(paste0(CIDER_PROJ_DIR, "cpi_ready.rds")) |>
  transmute(sampleId, gp, cider_score_depth_adj,
            immuneLENS_fraction = immuneLENS_fraction * 100) |>
  drop_na(immuneLENS_fraction)


# 1 - Main figure: T-cell validation — RNA and ImmuneLENS

Two-panel validation of the CIDER T-cell Score:
- Left: correlation with RNA T-cell gene expression signature
- Right: correlation with ImmuneLENS WGS-based T-cell fraction

In [ ]:
# ── Panel A: CIDER T-cell Score vs RNA T-cell signature ──────────────────────
# Mean AdjTPM is right-skewed; log10p1 on both axes linearises the relationship.
n_rna <- n_distinct(ready_rna$sampleId[ready_rna$gp == "T-cell"])
p_rna <- ready_rna |>
  filter(gp == "T-cell") |>
  dplyr::mutate(
    # Wider alpha_range (default 0.08-0.85) -> stronger fade: sparse points
    # nearly disappear, dense regions read as solidly opaque.
    pt_alpha = density_alpha(cider_score_depth_adj, rna,
                              tx = function(v) log10(v + 1),
                              ty = function(v) log10(v + 1),
                              alpha_range = c(0.03, 0.95))
  ) |>
  ggplot(aes(x = cider_score_depth_adj, y = rna)) +
  geom_point(aes(alpha = pt_alpha), size = 0.8) +
  scale_alpha_identity(guide = "none") +
  # Palette blue (COL_ACCENT) rather than ggplot's default blue -- Figure 2B.
  geom_smooth(method = "lm", colour = COL_ACCENT) +
  scale_x_continuous(trans = log10p1_trans) +
  scale_y_continuous(trans = log10p1_trans) +
  go_theme +
  theme(plot.subtitle = element_text(hjust = 0.5)) +
  labs(
    title    = "RNA Validation",
    subtitle = paste0("All Hartwig patients with RNA-seq, n = ", format(n_rna, big.mark = ",")),
    x        = "CIDER T-cell Score",
    y        = "RNA T-cell (CD3D, CD3E, CD3G)"
  ) +
  tags_main

# ── Panel B: CIDER T-cell Score vs ImmuneLENS T-cell fraction ────────────────
# ImmuneLENS % is on a linear scale — no log transform on x-axis.
# Faceted by T-cell overall and the three T-cell loci (TRA/TRD, TRB, TRG).
TCELL_LOCI       <- c("T-cell", "T-cell (TRA/TRD)", "T-cell (TRB)", "T-cell (TRG)")
TCELL_LOCI_LABEL <- c("T-cell (mean of loci)", "T-cell (TRA/TRD)", "T-cell (TRB)", "T-cell (TRG)")

n_il <- n_distinct(ready_il$sampleId[ready_il$gp == "T-cell"])
p_il <- ready_il |>
  filter(gp %in% TCELL_LOCI) |>
  mutate(gp = factor(recode(gp, "T-cell" = "T-cell (mean of loci)"), levels = TCELL_LOCI_LABEL)) |>
  dplyr::group_by(gp) |>
  dplyr::mutate(
    # Per-facet density in log10(x+1) × linear(y) space. Wider alpha_range
    # (default 0.08-0.85) -> stronger fade, matching the RNA panel above.
    pt_alpha = density_alpha(cider_score_depth_adj, immuneLENS_fraction,
                              tx = function(v) log10(v + 1),
                              alpha_range = c(0.03, 0.95))
  ) |>
  dplyr::ungroup() |>
  ggplot(aes(x = cider_score_depth_adj, y = immuneLENS_fraction)) +
  geom_point(aes(alpha = pt_alpha), size = 0.6) +
  scale_alpha_identity(guide = "none") +
  # Palette blue (COL_ACCENT) rather than ggplot's default blue -- Figure 2C.
  geom_smooth(method = "lm", colour = COL_ACCENT) +
  facet_wrap(~ gp, nrow = 1, scales = "free_x") +
  go_theme +
  theme(plot.subtitle = element_text(hjust = 0.5)) +
  labs(
    title    = "ImmuneLENS Validation",
    subtitle = paste0("All Hartwig CPI patients with WGS, n = ", format(n_il, big.mark = ",")),
    x        = "CIDER T-cell Score",
    y        = "ImmuneLENS T-cell Fraction"
  ) +
  tags_main

# Save individual panels for Figure 2B and 2C assembly
saveRDS(p_rna, paste0(CIDER_PROJ_DIR, "figs/tcell_validation_rna.rds"))
saveRDS(p_il,  paste0(CIDER_PROJ_DIR, "figs/tcell_validation_il.rds"))

# Also save combined for reference
tcell_validation_main <- (p_rna | p_il) +
  plot_layout(widths = c(1, 2)) +
  figure_annotation("CIDER T-cell Score Validation: RNA and ImmuneLENS")

set_plot_dims(14, 4.5); tcell_validation_main

# 2 - Supplementary: per-locus RNA and ImmuneLENS

Per-locus breakdown for T-cell loci (TRA/TRD, TRB, TRG).
RNA panels use unique CDR3 counts; ImmuneLENS panels use locus-specific fractions.


In [ ]:
# ── RNA per-locus ─────────────────────────────────────────────────────────────
rna_loci <- ready_rna |>
  filter(grepl("T-cell \\(", gp)) |>
  dplyr::group_by(gp) |>
  dplyr::mutate(
    pt_alpha = density_alpha(cider_score_depth_adj, rna,
                              tx = function(v) log10(v + 1),
                              ty = function(v) log10(v + 1))
  ) |>
  dplyr::ungroup() |>
  ggplot(aes(x = cider_score_depth_adj, y = rna)) +
  geom_point(aes(alpha = pt_alpha), size = 0.6) +
  scale_alpha_identity(guide = "none") +
  geom_smooth(method = "lm") +
  facet_wrap(~ gp, scales = "free", nrow = 1) +
  scale_x_continuous(trans = log10p1_trans) +
  stat_cor(aes(label = after_stat(r.label)), method = "pearson", size = 3.0) +
  go_theme +
  labs(
    title = "RNA vs CIDER Score — Per Locus",
    x     = "CIDER Score",
    y     = "RNA Expression (mean AdjTPM)"
  ) +
  tags_supp

# ── ImmuneLENS per-locus ───────────────────────────────────────────────────────
# ImmuneLENS % is linear — no log transform on x-axis.
il_loci <- ready_il |>
  filter(grepl("T-cell \\(", gp)) |>
  dplyr::group_by(gp) |>
  dplyr::mutate(
    pt_alpha = density_alpha(cider_score_depth_adj, immuneLENS_fraction,
                              tx = function(v) log10(v + 1))
  ) |>
  dplyr::ungroup() |>
  ggplot(aes(x = cider_score_depth_adj, y = immuneLENS_fraction)) +
  geom_point(aes(alpha = pt_alpha), size = 0.6) +
  scale_alpha_identity(guide = "none") +
  geom_smooth(method = "lm") +
  facet_wrap(~ gp, scales = "free", nrow = 1) +
  stat_cor(aes(label = after_stat(r.label)), method = "pearson", size = 3.0) +
  go_theme +
  labs(
    title = "ImmuneLENS vs CIDER Score — Per Locus",
    x     = "CIDER Score",
    y     = "ImmuneLENS T-cell (%)"
  ) +
  tags_supp

tcell_validation_supplement <- (rna_loci / il_loci) +
  figure_annotation()

set_plot_dims(11, 8); tcell_validation_supplement

# 3 - Full ImmuneLENS validation — all available groups

Faceted scatter plot of CIDER Score vs ImmuneLENS fraction
for every group with ImmuneLENS data: T-cell (overall) and the three T-cell
loci (TRA/TRD, TRB, TRG).
Pearson r shown per facet. No log transform on x-axis — ImmuneLENS fraction is linear.


In [ ]:
# Group order: T-cell overall first, then its three loci
IL_GROUP_ORDER       <- c("T-cell", "T-cell (TRA/TRD)", "T-cell (TRB)", "T-cell (TRG)")
IL_GROUP_ORDER_LABEL <- c("T-cell (mean of loci)", "T-cell (TRA/TRD)", "T-cell (TRB)", "T-cell (TRG)")

il_all_groups <- ready_il |>
  filter(gp %in% IL_GROUP_ORDER) |>
  mutate(gp = factor(recode(gp, "T-cell" = "T-cell (mean of loci)"), levels = IL_GROUP_ORDER_LABEL))

n_il_full <- n_distinct(il_all_groups$sampleId)
p_il_full <- il_all_groups |>
  dplyr::group_by(gp) |>
  dplyr::mutate(
    pt_alpha = density_alpha(cider_score_depth_adj, immuneLENS_fraction,
                              tx = function(v) log10(v + 1))
  ) |>
  dplyr::ungroup() |>
  ggplot(aes(x = cider_score_depth_adj, y = immuneLENS_fraction)) +
  geom_point(aes(alpha = pt_alpha), size = 0.6) +
  scale_alpha_identity(guide = "none") +
  geom_smooth(method = "lm") +
  facet_wrap(~ gp, nrow = 2, ncol = 3, scales = "free") +
  go_theme +
  theme(plot.subtitle = element_text(hjust = 0.5)) +
  labs(
    title    = "ImmuneLENS Validation — All Groups",
    subtitle = paste0("All Hartwig CPI patients with WGS, n = ", format(n_il_full, big.mark = ",")),
    x        = "CIDER Score",
    y        = "ImmuneLENS (%)"
  ) +
  tags_supp

set_plot_dims(14, 4); p_il_full
saveRDS(p_il_full, paste0(CIDER_PROJ_DIR, "figs/il_validation_all_groups.rds"))

# 4 - RNA validation — T-cell

Faceted scatter of CIDER Score vs RNA expression for the T-cell group.
Uses `log10p1` transform on x-axis. Pearson r shown per panel.


In [ ]:
# ── RNA validation: all three broad groups ───────────────────────────────────
n_rna_broad <- n_distinct(ready_rna$sampleId[ready_rna$gp == "T-cell"])
rna_broad_groups <- ready_rna |>
  filter(gp == "T-cell") |>
  mutate(
    gp       = factor(gp, levels = "T-cell"),
    pt_alpha = density_alpha(cider_score_depth_adj, rna,
                              tx = function(v) log10(v + 1),
                              ty = function(v) log10(v + 1))
  ) |>
  ggplot(aes(x = cider_score_depth_adj, y = rna)) +
  geom_point(aes(alpha = pt_alpha), size = 0.8) +
  scale_alpha_identity(guide = "none") +
  geom_smooth(method = "lm") +
  facet_wrap(~ gp, scales = "free", nrow = 1) +
  scale_x_continuous(trans = log10p1_trans) +
  scale_y_continuous(trans = log10p1_trans) +
  go_theme +
  theme(plot.subtitle = element_text(hjust = 0.5)) +
  labs(
    title    = "CIDER Score vs RNA Expression — Broad Groups",
    subtitle = paste0("All Hartwig patients with RNA-seq, n = ", format(n_rna_broad, big.mark = ",")),
    x        = "CIDER Score",
    y        = "RNA Expression (mean AdjTPM)"
  ) +
  tags_supp

set_plot_dims(10, 4); rna_broad_groups
saveRDS(rna_broad_groups, paste0(CIDER_PROJ_DIR, "figs/rna_validation_broad.rds"))

# 5 - Pairwise correlation heatmap — all T-cell loci

4×4 Pearson correlation matrix across all available patients.
Rows and columns ordered by `GP_LEVELS` (T-cell overall → its three loci).


In [ ]:
# ── Wide format: one row per patient, one column per locus ──────────────────
# T-cell overall + its three loci
tcell_levels <- GP_LEVELS[grepl("T-cell", GP_LEVELS)]
tcell_labels <- dplyr::recode(tcell_levels,
  "T-cell"         = "T-cell",
  "T-cell (TRA/TRD)" = "TRA/TRD",
  "T-cell (TRB)"   = "TRB",
  "T-cell (TRG)"   = "TRG"
)

ready_wide_corr <-
  readRDS(paste0(CIDER_PROJ_DIR, "ready.rds")) |>
  filter(grepl("T-cell", gp)) |>
  select(sampleId, gp, cider_score_depth_adj) |>
  pivot_wider(names_from = gp, values_from = cider_score_depth_adj,
              values_fn = mean) |>
  select(sampleId, dplyr::any_of(tcell_levels))

# ── Pearson correlation matrix ────────────────────────────────────────────────
cor_mat <- ready_wide_corr |>
  select(-sampleId) |>
  cor(use = "pairwise.complete.obs", method = "pearson")

# ── Long format for ggplot — lower triangle only ─────────────────────────────
cor_long <- cor_mat |>
  as.data.frame() |>
  tibble::rownames_to_column("row") |>
  tidyr::pivot_longer(-row, names_to = "col", values_to = "r") |>
  mutate(
    row = factor(dplyr::recode(row, !!!setNames(tcell_labels, tcell_levels)), levels = rev(tcell_labels)),
    col = factor(dplyr::recode(col, !!!setNames(tcell_labels, tcell_levels)), levels = tcell_labels)
  ) |>
  # Keep only lower triangle (including diagonal)
  filter(as.integer(row) <= (length(tcell_levels) - as.integer(col) + 1))

# ── Heatmap ───────────────────────────────────────────────────────────────────
n_corr <- nrow(ready_wide_corr)
corr_heatmap <- cor_long |>
  ggplot(aes(x = col, y = row, fill = r)) +
  geom_tile(colour = "white", linewidth = 0.5) +
  geom_text(aes(label = round(r, 2)), size = 3.5, fontface = "bold") +
  # Continuous white -> COL_ACCENT (blue) gradient for correlation strength
  # (0-1).
  scale_fill_gradient2(
    low      = "white",
    high     = COL_ACCENT,
    limits   = c(0, 1),
    name     = "Pearson r",
    guide    = "none"
  ) +
  go_theme +
  theme(
    plot.subtitle    = element_text(hjust = 0.5),
    axis.text.x      = element_text(size = 9),
    axis.text.y      = element_text(size = 9),
    panel.border     = element_blank(),
    axis.line        = element_blank(),
    legend.position  = "right"
  ) +
  labs(
    title    = "Pairwise Correlations between Loci",
    subtitle = paste0("All Hartwig patients with WGS; n = ", format(n_corr, big.mark = ",")),
    x        = NULL,
    y        = NULL
  )

set_plot_dims(7, 6); corr_heatmap
saveRDS(corr_heatmap, paste0(CIDER_PROJ_DIR, "figs/cider_corr_heatmap.rds"))